In [76]:
import pandas as pd
import seaborn as sn
import numpy as np
import matplotlib.pyplot as plt

Lectura:
- Explicación de aumento de temperatura:https://berkeley-earth-wp-offload.storage.googleapis.com/wp-content/uploads/2022/12/03232411/skeptics-guide-to-climate-change-1.pdf
- Temperaturas de japon + analisis con volcan: https://berkeleyearth.org/temperature-location/36.17N-139.23E

Notamos que se usa el MA12 para quitar inestabilidad a las temperaturas, el cuál se calcularía la temperatura actual con las 11 temperaturas anteriores promediadas

# Funciones utiles

Se cambian las columnas de latitud y longitud que contienen caracteres no numericos. La siguiente funció n transforma el caracter final del string (N,S,E o W) a +1 o -1 y lo multiplica por la parte numerica del string.

In [77]:

def coord_to_float(s: str) -> float:
    sign = -1 if s[-1] in ['S', 'W'] else 1
    return sign * float(s[:-1])

# Carga de datos

In [78]:
MajorCityCsv = './datasets/GlobalLandTemperaturesByMajorCity.csv'

In [79]:
# engine: para seleccionar la libreria que abre el archivo
# pyarrow para archivos muy grandes
# dtype_backend: como se almacenan los datos
MajorCityDf = pd.read_csv(
    MajorCityCsv,
    dtype_backend='pyarrow',
    engine='pyarrow'
)

# Preparación de datos

In [80]:
MajorCityDf.info()

<class 'pandas.DataFrame'>
RangeIndex: 239177 entries, 0 to 239176
Data columns (total 7 columns):
 #   Column                         Non-Null Count   Dtype               
---  ------                         --------------   -----               
 0   dt                             239177 non-null  date32[day][pyarrow]
 1   AverageTemperature             228175 non-null  double[pyarrow]     
 2   AverageTemperatureUncertainty  228175 non-null  double[pyarrow]     
 3   City                           239177 non-null  string[pyarrow]     
 4   Country                        239177 non-null  string[pyarrow]     
 5   Latitude                       239177 non-null  string[pyarrow]     
 6   Longitude                      239177 non-null  string[pyarrow]     
dtypes: date32[day][pyarrow](1), double[pyarrow](2), string[pyarrow](4)
memory usage: 14.3 MB


In [81]:
print(MajorCityDf.duplicated().sum()) # buscar datos duplicados
print(MajorCityDf.isna().sum()) # buscar valores NaN

0
dt                                   0
AverageTemperature               11002
AverageTemperatureUncertainty    11002
City                                 0
Country                              0
Latitude                             0
Longitude                            0
dtype: int64


In [82]:
cols = ['AverageTemperature', 'AverageTemperatureUncertainty']
MajorCityDf[cols] = (
    MajorCityDf
    .groupby('City')[cols]
    .apply(lambda s: s.interpolate(method='linear', limit_direction='both'))
    .reset_index(level=0, drop=True)
)

In [83]:
print(MajorCityDf.isna().sum())

dt                               0
AverageTemperature               0
AverageTemperatureUncertainty    0
City                             0
Country                          0
Latitude                         0
Longitude                        0
dtype: int64


In [84]:
# se crea esta columna para obtener un valor promedio por año de las temperaturas
# columna.dt.year = extrae el año
MajorCityDf['year'] = MajorCityDf.dt.dt.year

# Y esta otra lo mismo pero por mes
# columna.dt.month
MajorCityDf['month'] = MajorCityDf.dt.dt.month

# Cambia el nombre de un pais por uno mas corto
MajorCityDf['Country'] = MajorCityDf['Country'].replace('Congo (Democratic Republic Of The)', 'Congo')

# Cambiamos el formato de las coordenadas para poder usarlas mas adelante
MajorCityDf = MajorCityDf.sort_values('dt').reset_index().drop(['index'], axis=1)
MajorCityDf['Latitude'] = MajorCityDf['Latitude'].apply(coord_to_float)
MajorCityDf['Longitude'] = MajorCityDf['Longitude'].apply(coord_to_float)

Nuevo tipo de grafico suavizado con una media movil de 12 meses (cada mes es el promedio de ese vs los 11 anteriores)

In [85]:
MajorCityDf["MA12"] = (
    MajorCityDf
    .groupby(["Country", "City"])["AverageTemperature"]
    .transform(lambda x: x.rolling(12, min_periods=1, center=True).mean())
)
MajorCityDf['dt'] = pd.to_datetime(MajorCityDf['dt'])

### Cruce
Se cruzan los datos para identificar las ciudades por continente. Esto permitirá hacer mejores gráficos

In [86]:
# Fuente https://www.kaggle.com/datasets/hserdaraltan/countries-by-continent

country_by_continent = pd.read_csv(
    './datasets/countries_by_continents.csv',
    dtype_backend='pyarrow',
    engine='pyarrow'
)

In [87]:
MajorCityDf = MajorCityDf.merge(
    country_by_continent,
    how='left',
    on='Country'
)

In [88]:
MajorCityDf[MajorCityDf['Continent'].isna()]['Country'].unique()

<ArrowExtensionArray>
['Burma', "Côte D'Ivoire"]
Length: 2, dtype: string[pyarrow]

In [89]:
MajorCityDf.loc[MajorCityDf['Country'] == 'Burma', 'Continent'] = 'Asia'
MajorCityDf.loc[MajorCityDf['Country'] == "Côte D'Ivoire", 'Continent'] = 'Africa'
MajorCityDf.loc[MajorCityDf['Country'] == "Congo (Democratic Republic Of The)", 'Continent'] = 'Africa'